In [ ]:
# %pip install -q youtube-transcript-api langchain-community langchain-huggingface faiss-cpu tiktoken python-dotenv sentence-transformers


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [132]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv

## Step 1 - a - Indexing(Document loading)

In [133]:
video_id="9vM4p9NN0Ts" ##only the ID and not full URL

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)
    
except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

so let's get started uh so I'll be talking about building llms today um so I think a lot of you have heard of llms before uh but just as a quick recap uh llms standing for large language models are basically all the chat Bots uh that you've been hearing about recently so uh Chad GPT from open ey Claud from entropic Gemini and and lman other type of models like this and today we'll be talking about how do they actually work so it's going to be an overview because it's only one lecture and it's hard to compress everything but hopefully I'll touch a little bit about all the components that are needed to train uh some of these llms uh also if you have questions please interrupt me and ask uh if you have a question most likely other people in the room or on Zoom have other have the same question so please ask um great so what matters when training llms um so there a few key components that matter uh one is the architecture so as you probably all know LMS are newal networks and when you thin

## Step 1 - b - Indexing(Text Splitting)

In [134]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitter.create_documents([transcript])
len(chunks)

126

## Step - 1 - c and d - Indexing(Embedding Generation and storing vector in vector store)

In [135]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store=FAISS.from_documents(chunks,embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8265.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [136]:
vector_store.index_to_docstore_id

{0: '872d6f22-14ee-4002-8cf7-cbd4d2c53834',
 1: 'e4b69ca8-8cb2-4827-b3ed-8de556841552',
 2: '1577b7fc-0e4c-480e-9041-15073cccbfcd',
 3: 'f96f125c-291c-41a2-861b-e5fd3f3371a6',
 4: '48fcbc3c-1e6e-4c0b-8431-7700e53ad88c',
 5: '15664721-10c7-4cbf-9dbf-bb1419e049d0',
 6: '31aa24ba-e4de-4bc4-812b-1890c399a938',
 7: '7a976889-129c-4358-bbc0-801de6b8eaed',
 8: 'aad54a35-e18a-43e4-8090-664c44732f94',
 9: '0d7755ec-504d-4336-936e-1c4cd9a9456c',
 10: '3289b926-e781-4d57-b7bb-f353fe6b15f2',
 11: '5ce0fbbb-f0db-4305-b79a-0158275b8b08',
 12: '21d80bc4-6fbb-431e-8009-1c4ae5a746e0',
 13: 'a84df2d5-5384-470e-be5c-5dcfbb2bcead',
 14: '4627c708-626a-4663-91fd-0a8e096c982a',
 15: 'e61dcd2a-b2ae-4958-b72c-3fbdfeefff4f',
 16: '25772e80-2a87-45a4-869f-933494f42b60',
 17: 'fe4136c1-dae2-4a52-88c7-3cc1eca505c4',
 18: '0c5c3d8a-7332-4610-ab35-9f941596eb70',
 19: 'cf5a0445-7258-4c2c-b47e-9c8814f167b9',
 20: 'f9af2007-3b1c-4117-8c54-faea6921ba50',
 21: '002caf01-9004-4d3e-8c2e-dc6624eafdd3',
 22: '90f53b83-bcdc-

## Step - 2 - Retrieval

In [137]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [138]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x114531b50>, search_kwargs={'k': 4})

In [139]:
retriever.invoke('what is deepmind')

[Document(id='5bea24d1-e5f2-4f39-90ee-97d901482353', metadata={}, page_content="about the relevance of the term monopsony and then it says monopsony refers to a market structure blah blah blah and that's a human that wrote that um so actually this is open Assistant which was a a way to collect um uh data online by humans so this type of supervised fine tuning or alignment is really the key of Chad GPT this is what made uh the big jump from gpt3 which was mostly something that was known by AI researchers to Chad GPT which became known by basically everyone um so the problem with uh human data is that it's uh very slow to collect and very expensive um so one possible simple idea is to use llms to scale data collection uh so that's exactly what we did with alpaca uh one year ago what we did is that we asked uh humans or we use a data set of human uh question answers so there were 175 uh question answers here and we asked the best mod at the time so text3 to basically generate many more of

## Step - 3 - Augmentation

In [140]:
prompt=PromptTemplate(
    template= """
    You are a helpful assistant.
    Answer ONLY from the provided transcript context,
    if the context is insufficient, just say you dont know.

    {context}
    Question:{question}
    """,
    input_variables=['context','question']
)

In [146]:
question= ' is topic of llms discussed in this video? adn what is it ?'
retrieved_docs=retriever.invoke(question)
retrieved_docs

[Document(id='872d6f22-14ee-4002-8cf7-cbd4d2c53834', metadata={}, page_content="so let's get started uh so I'll be talking about building llms today um so I think a lot of you have heard of llms before uh but just as a quick recap uh llms standing for large language models are basically all the chat Bots uh that you've been hearing about recently so uh Chad GPT from open ey Claud from entropic Gemini and and lman other type of models like this and today we'll be talking about how do they actually work so it's going to be an overview because it's only one lecture and it's hard to compress everything but hopefully I'll touch a little bit about all the components that are needed to train uh some of these llms uh also if you have questions please interrupt me and ask uh if you have a question most likely other people in the room or on Zoom have other have the same question so please ask um great so what matters when training llms um so there a few key components that matter uh one is the a

In [147]:
#joining the top 4 similar retrieved docs
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [148]:
final_prompt=prompt.invoke({'context':context_text,'question':question})

In [149]:
load_dotenv()

llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-safeguard-20b",
    task="text-generation"
)

model=ChatHuggingFace(llm=llm)

In [150]:
answer=model.invoke(final_prompt)
print(answer)

content='Yes – the video is all about **LLMs (Large Language Models)**.  \n\nIt explains that LLMS are neural‑network models (mostly based on Transformers) and gives an overview of the key parts involved in building and training them:  \n\n1. **Architecture** – the neural‑network design (typically a Transformer).  \n2. **Training loss and algorithm** – how the model learns from data.  \n3. **Data** – the large corpora used to train the model.  \n4. **Evaluation** – metrics and tests to gauge progress.  \n5. **Systems** – the hardware and software needed to run such large models efficiently.  \n\nIt also touches on **alignment**, the post‑training process that shapes the model’s behavior so it follows user instructions and avoids unwanted outputs.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 238, 'prompt_tokens': 906, 'total_tokens': 1144}, 'model_name': 'openai/gpt-oss-safeguard-20b', 'system_fingerprint': 'fp_e3febdc4be', 'finish_reason': 'stop', 'logp

# BUILDING A CHAIN

In [151]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [152]:
def format_docs(retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [153]:
parallel_chain=RunnableParallel({
    'context':retriever | RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [154]:
parser=StrOutputParser()

In [155]:
main_chain=parallel_chain | prompt | llm | parser

In [156]:
main_chain.invoke('summarize the video')

ValueError: Task 'text-generation' not supported for provider 'groq'. Available tasks: ['conversational']